In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.streaming import StreamingQuery
from typing import Optional

import logging

In [0]:

def load_autoloader(
    topic: str,
    schema_location: str,
    format: str = "parquet"
) -> DataFrame:
    """Load new files for a topic incrementally using Auto Loader."""
    base_source_path = "/Volumes/workspace/jobs/files"
    source_path = f"{base_source_path}/{topic}"

    df = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", format)
        .option("cloudFiles.schemaEvolutionMode", "rescue")
        .option("cloudFiles.schemaLocation", f"{schema_location}/schema/{topic}")
        .load(source_path)
        )
    return df
    

def write_to_delta(
    df: DataFrame,
    topic: str,
    target_table: str,
    checkpoint: str,
    mode: str = "append",
):
    """Write a streaming DataFrame to a Delta table with a per-topic checkpoint."""
    return (
        df.writeStream.format("delta")
        .outputMode(mode)
        .option("checkpointLocation", f"{checkpoint}/checkpoints/{topic}")
        .trigger(availableNow=True)
        .toTable(target_table)
    )


def ingestion_pipeline(
    topic: str,
    checkpoint: str,
    target_table: str,
    mode: str = "append",
    format: str = "parquet"
):
    """Run the full pipeline for a topic: load with Auto Loader and write to Delta."""
    stream_df = load_autoloader(
        topic=topic,
        schema_location=checkpoint,
        format=format,
    )

    return write_to_delta(
        df=stream_df,
        topic=topic,
        target_table=target_table,
        mode=mode,
        checkpoint=checkpoint
    )

In [0]:
logger = logging.getLogger(__name__)

ingestion_config = [
    {
        "topic": "job_postings",
        "checkpoint": "/Volumes/workspace/jobs/auto_loader_metadata",
        "target_table": "workspace.jobs.job_postings"
    },
    {
        "topic": "job_skills",
        "checkpoint": "/Volumes/workspace/jobs/auto_loader_metadata",
        "target_table": "workspace.jobs.job_skills"
    }
]

for config in ingestion_config:
    logger.info(f"Loading {config['topic']} to {config['target_table']}")
    streaming_query = ingestion_pipeline(**config)

    streaming_query.awaitTermination()
    logger.info(f"Finished loading {config['topic']} to {config['target_table']}")